In [ ]:
import gc
import os
import time

import numpy as np
import optuna
import pandas as pd
import torch
from pytorch_tabular import TabularModel
from pytorch_tabular.config import (
    DataConfig,
    OptimizerConfig,
    TrainerConfig,
)
from pytorch_tabular.models import (
    CategoryEmbeddingModelConfig,
    GANDALFConfig,
)
from pytorch_tabular.models.common.heads import LinearHeadConfig
from pytorch_tabular.utils import get_balanced_sampler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Hyperparameter tuning -- GANDALF, Category Embedding

Tunes the two `pytorch_tabular` feature models, GANDALF and Category
Embedding, using the same validation protocol, objective and trial budget
as `07a_tuning_features_ml.ipynb` and `07d_tuning_timeseries_dl.ipynb`.

TabPFN is tuned in `07c_tuning_features_tabpfn.ipynb`, because it needs a
different environment (`tabpfn`) from `pytorch_tabular` (`torch_tabular`).
GANDALF and Category Embedding can be re-run independently of each other;
outputs are merged into the shared CSVs (see "Save outputs").

Both models use the `TabularModel`/`DataConfig`/`TrainerConfig`/
`OptimizerConfig`/`LinearHeadConfig` setup from
`05b_feature_classification_dl.ipynb`, with an Optuna search before the
final fit.

## Validation protocol

Uses the 5 stratified-group CV folds per dataset
(`indices_cv_fold_lookup.csv`). Each outer fold i is the **test** fold in
turn. For the search, the next `N_INNER_FOLDS` = 2 folds,
(i+1) mod 5 and (i+2) mod 5, are each used as an inner validation fold,
with the other 3 non-test folds as inner training data. Each Optuna trial
is scored by the mean `f1_macro` over these 2 inner validation folds; the
test fold is never used in the search.

The winning setting is then retrained on the 4 non-test folds and
evaluated once on the test fold. A grouped, stratified subset of about 1/7
of the training parent bursts (`ES_SPLITS = 7`, `early_stopping_split`, as
in `05b`) is held out of the 4 non-test folds for early stopping, and the
model is fit on the rest.

Early stopping never uses the test fold. In the search, each inner fit
early-stops on the inner validation fold it is scored on, so trial scores
are slightly optimistic; this bias is shared by all trials (including the
enqueued default) and does not touch the test fold. Every fit uses
`TrainerConfig(seed=RANDOM_STATE)`. Batch sizes are set
with `safe_batch_size` so no fit has a final batch of one sample (which
crashes `pytorch_tabular`).

## Search space

`get_balanced_sampler` (`use_class_weight=True`) is fixed for both models.
Each model gets 2 tuned parameters, one for capacity and one for
regularization:

- **GANDALF**: `gflu_stages` (5-25, capacity: number of Gated Feature
  Learning Unit blocks) + `gflu_dropout` (0.0-0.5, regularization).
  `learnable_sparsity=False` is fixed, as in `05b`.
- **Category Embedding**: `layers` (capacity, categorical over
  `CATEGORY_EMBEDDING_LAYER_PRESETS`: five hyphen-joined width strings,
  ordered by size, with the default `"1024-512-256"` in the middle) +
  `dropout` (0.0-0.5, regularization: the backbone dropout, separate from
  the head's fixed `dropout=0.1`). `layers` is a string, so it is searched
  with `suggest_categorical`, which TPE treats as unordered.

## Default control (validation side)

`GANDALF_DEFAULT_PARAMS` (`gflu_stages=15, gflu_dropout=0.0`) and
`CATEGORY_EMBEDDING_DEFAULT_PARAMS` (`layers="1024-512-256", dropout=0.0`)
are the configurations used in `05b`. Both lie inside the search space, so
each is enqueued as one of the `N_TRIALS` trials via `study.enqueue_trial`
(not an extra trial). Its score is saved as `default_val_f1_macro`,
directly comparable to `val_f1_macro`.

## Default control (test side)

Each `*_DEFAULT_PARAMS` is also retrained and evaluated on the same
outer fold as the tuned winner, through the same code path
(`retrain_and_evaluate_ptab`), with the same early-stopping split and
seed, so tuned vs default on the test fold is like-for-like. These rows
(`model="<model>_default"`) are saved to separate
`behaviour_features_dl_tuned_default_control_*.csv` files, so the
`..._tuned_metrics.csv` / `..._tuned_confusion.csv` files read by
`12b_publication_plots_fig3.R` contain only tuned rows.

## Objective and budget

`f1_macro` averaged over the inner validation folds, Optuna TPE sampler
(`multivariate=True`, seed `RANDOM_STATE`), `N_TRIALS` = 50 per outer
fold, the same budget as the other tuning notebooks.

## Scope

One dataset, Burst_1/uncorrected (`DATASETS`), x 5 outer folds = 5 Optuna
studies per model. Only the behaviour (multiclass) target is run.



## Load the main CV fold lookup and set tuning constants

In [ ]:
FOLD_LOOKUP_PATH = "../data/temp/indices_cv_fold_lookup.csv"
DATASETS = [
    ("burst_1", "uncorrected"),
]
N_TRIALS = 50
RANDOM_STATE = 42
SEED = RANDOM_STATE  # early-stopping split seed in the retrains
ES_SPLITS = 7  # holds out ~1/7 (~14%) of the training parents for early stopping
# Number of inner validation folds whose f1_macro is averaged per Optuna
# trial (folds (i+1) and (i+2) mod 5 for outer test fold i).
N_INNER_FOLDS = 2
# Default control: GANDALF configuration from
# 05b_feature_classification_dl.ipynb (gflu_stages=15, gflu_dropout left at
# pytorch_tabular's default of 0.0). Inside the search space, so enqueued
# via study.enqueue_trial.
GANDALF_DEFAULT_PARAMS = {"gflu_stages": 15, "gflu_dropout": 0.0}
# Default control: Category Embedding configuration from
# 05b_feature_classification_dl.ipynb (layers="1024-512-256", backbone
# dropout left at pytorch_tabular's default of 0.0). Inside the search
# space, so enqueued the same way.
CATEGORY_EMBEDDING_DEFAULT_PARAMS = {"layers": "1024-512-256", "dropout": 0.0}
# Category Embedding's capacity axis: "layers" is a hyphen-joined width
# string, so it is searched as a categorical over these five presets
# (two smaller and two larger than the default "1024-512-256").
CATEGORY_EMBEDDING_LAYER_PRESETS = [
    "256-128",
    "512-256-128",
    "1024-512-256",
    "1024-512-256-128",
    "2048-1024-512-256",
]

fold_lookup_all = pd.read_csv(FOLD_LOOKUP_PATH)
print(
    f"Loaded fold lookup: {len(fold_lookup_all)} rows, "
    f"bursts={sorted(fold_lookup_all['burst'].unique())}"
)

## GANDALF + Category Embedding shared pytorch_tabular trainer

Both models are trained via `pytorch_tabular`'s `TabularModel` and share
one trainer: `eval_ptab_cv_folds` for the search and
`retrain_and_evaluate_ptab` for the final retrain. The only difference at
fit time is the `*Config` object, passed in via
`model_config_fn(common_params)`. The trainer setup and the
`early_stopping_split`/`safe_batch_size` helpers are the same as in
`05b_feature_classification_dl.ipynb`.

In [ ]:
def early_stopping_split(y_train, groups_train, n_classes, seed=SEED, max_tries=20):
    """Grouped, stratified split of the TRAINING folds only (same as 05b). The
    test fold is never seen during training, early stopping or checkpoint
    selection.

    Re-draws (seed, seed+1, ...) until every class is in the fitting split,
    because pytorch_tabular silently drops the output of an unseen class. A
    class may still be absent from the early-stopping subset; that only removes
    it from the stopping loss.

    Returns:
        tuple: (fit_idx, es_idx) positional indices into the training set
    """
    y_train, groups_train = np.asarray(y_train), np.asarray(groups_train)
    if len(np.unique(y_train)) < n_classes:
        raise ValueError("A class is missing from the training folds themselves")
    for k in range(max_tries):
        sgkf = StratifiedGroupKFold(
            n_splits=ES_SPLITS, shuffle=True, random_state=seed + k
        )
        fit_idx, es_idx = next(
            sgkf.split(np.zeros(len(y_train)), y_train, groups_train)
        )
        if len(np.unique(y_train[fit_idx])) == n_classes:
            if k > 0:
                print(
                    f"  early-stopping split re-drawn {k} time(s) to keep all classes"
                )
            return fit_idx, es_idx
    raise ValueError("No early-stopping split keeps every class in the fitting split")


def safe_batch_size(*sizes, bs=64):
    """Smallest batch size >= bs that leaves no final batch of a single sample in
    any of the given sets. A one-sample batch raises an IndexError inside
    pytorch_tabular."""
    while any(n % bs == 1 for n in sizes):
        bs += 1
    return bs


def _build_ptab_configs(batch_size=64):
    """Build the trainer/head/optimizer configs shared by every
    pytorch_tabular fit in this notebook (same settings as
    05b_feature_classification_dl.ipynb).

    Args:
        batch_size (int): Training batch size, from safe_batch_size for
            the fit it is used in.

    Returns:
        tuple: (trainer_config, optimizer_config, common_params) --
            common_params is the task/head/head_config dict passed to
            every *Config (GANDALFConfig/CategoryEmbeddingModelConfig).
    """
    trainer_config = TrainerConfig(
        auto_lr_find=True,
        batch_size=batch_size,
        max_epochs=100,
        early_stopping="valid_loss",
        early_stopping_mode="min",
        early_stopping_min_delta=0.001,
        early_stopping_patience=10,
        checkpoints="valid_loss",
        load_best=True,
        track_grad_norm=2,
        progress_bar="none",
        accelerator="gpu",
        seed=RANDOM_STATE,
    )
    head_config = LinearHeadConfig(
        layers="",
        dropout=0.1,
        initialization="kaiming",
    ).__dict__
    optimizer_config = OptimizerConfig(
        lr_scheduler="CosineAnnealingWarmRestarts",
        lr_scheduler_params={"T_0": 100, "T_mult": 1, "eta_min": 1e-5},
    )
    common_params = {
        "task": "classification",
        "head": "LinearHead",
        "head_config": head_config,
    }
    return trainer_config, optimizer_config, common_params


def eval_ptab_cv_folds(cv_folds, model_config_fn, n_classes, use_class_weight=True):
    """Fit+score one pytorch_tabular hyperparameter setting (built by
    model_config_fn) across all of cv_folds (the N_INNER_FOLDS inner
    train/val splits), returning the mean f1_macro.

    Used by both models' Optuna objectives. The inner validation fold is
    also pytorch_tabular's early-stopping set; the test fold is never used.

    Args:
        cv_folds (list): list of (X_train, y_train, X_val, y_val) tuples,
            one per inner fold -- y_train/y_val are integer-encoded
            (LabelEncoder), matching process_parquet_file_gandalf/
            process_parquet_file_category_embedding.
        model_config_fn (callable): common_params (dict) -> a
            GANDALFConfig/CategoryEmbeddingModelConfig instance built from
            one trial's suggested hyperparameters.
        n_classes (int): Number of classes; every inner training set must
            contain all of them.
    """
    scores = []
    for X_train, y_train, X_val, y_val in cv_folds:
        if len(np.unique(y_train)) < n_classes:
            raise ValueError("A class is missing from an inner training set")
        bs = safe_batch_size(len(X_train), len(X_val))
        trainer_config, optimizer_config, common_params = _build_ptab_configs(bs)
        model_config = model_config_fn(common_params)

        train_df = X_train.copy()
        val_df = X_val.copy()
        train_df["target"] = y_train
        val_df["target"] = y_val
        train_df.reset_index(drop=True, inplace=True)
        val_df.reset_index(drop=True, inplace=True)

        data_config = DataConfig(
            target=["target"],
            continuous_cols=list(X_train.columns),
            categorical_cols=[],
            pin_memory=True,
            num_workers=0,
        )
        train_sampler = (
            get_balanced_sampler(np.asarray(y_train).ravel())
            if use_class_weight
            else None
        )

        model = TabularModel(
            data_config=data_config,
            model_config=model_config,
            optimizer_config=optimizer_config,
            trainer_config=trainer_config,
            verbose=False,
        )
        model.fit(train=train_df, validation=val_df, train_sampler=train_sampler)

        pred_df = model.predict(val_df)
        y_pred = pred_df["prediction"].values
        scores.append(f1_score(y_val, y_pred, average="macro", zero_division=0))

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return float(np.mean(scores))


def retrain_and_evaluate_ptab(
    X_train_val,
    y_train_val,
    X_test,
    y_test,
    model_config_fn,
    param_cols,
    class_names,
    groups_train_val,
    use_class_weight=True,
):
    """Fit one pytorch_tabular hyperparameter setting (built by
    model_config_fn) on X_train_val/y_train_val and evaluate once on
    X_test/y_test, returning (metrics_row, confusion_rows) --
    confusion_rows is a list of dicts without a "model" key (the caller
    tags that).

    Used for both final retrains per outer fold (tuned winner and default
    control). The roc_auc/probability-column extraction is the same as in
    05b_feature_classification_dl.ipynb.

    Args:
        param_cols (dict): the hyperparameter values to prepend to
            metrics_row (e.g. {"gflu_stages": 15, "gflu_dropout": 0.0}).
        groups_train_val (np.ndarray): Parent-burst ID per X_train_val
            window, for the grouped early-stopping split. Early stopping
            uses a held-out subset of the non-test folds; the test fold is
            only scored.
    """
    n_classes = len(class_names)

    train_df = X_train_val.copy()
    test_df = X_test.copy()
    train_df["target"] = y_train_val
    test_df["target"] = y_test
    train_df.reset_index(drop=True, inplace=True)
    test_df.reset_index(drop=True, inplace=True)

    fit_idx, es_idx = early_stopping_split(y_train_val, groups_train_val, n_classes)
    fit_df = train_df.iloc[fit_idx].reset_index(drop=True)
    es_df = train_df.iloc[es_idx].reset_index(drop=True)
    bs = safe_batch_size(len(fit_df), len(es_df), len(test_df))
    trainer_config, optimizer_config, common_params = _build_ptab_configs(bs)
    model_config = model_config_fn(common_params)

    data_config = DataConfig(
        target=["target"],
        continuous_cols=list(X_train_val.columns),
        categorical_cols=[],
        pin_memory=True,
        num_workers=0,
    )
    train_sampler = (
        get_balanced_sampler(fit_df["target"].to_numpy()) if use_class_weight else None
    )

    t0 = time.time()
    model = TabularModel(
        data_config=data_config,
        model_config=model_config,
        optimizer_config=optimizer_config,
        trainer_config=trainer_config,
        verbose=False,
    )
    model.fit(train=fit_df, validation=es_df, train_sampler=train_sampler)

    # Test fold only scored, never used for stopping
    pred_df_with_proba = model.predict(test_df)
    y_pred = pred_df_with_proba["prediction"].values

    roc = np.nan
    prob_cols = [col for col in pred_df_with_proba.columns if "_probability" in col]
    if prob_cols:
        prob_cols.sort(key=lambda name: int(name.split("_")[0]))
        proba_values = pred_df_with_proba[prob_cols].values
        try:
            if proba_values.shape[1] == n_classes:
                roc = roc_auc_score(
                    y_test, proba_values, multi_class="ovr", average="weighted"
                )
        except ValueError:
            roc = np.nan

    metrics_row = {
        **param_cols,
        "roc_auc": roc,
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_acc": balanced_accuracy_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "precision": precision_score(
            y_test, y_pred, average="weighted", zero_division=0
        ),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "precision_macro": precision_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "time": time.time() - t0,
    }

    cm = confusion_matrix(y_test, y_pred, labels=np.arange(n_classes))
    confusion_rows = [
        {
            "actual_label": class_names[i],
            "predicted_label": class_names[j],
            "count": cm[i][j],
        }
        for i in range(n_classes)
        for j in range(n_classes)
    ]

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics_row, confusion_rows


## GANDALF Optuna search + final evaluation, for one outer test fold

In [ ]:
def make_gandalf_objective(cv_folds, n_classes, use_class_weight=True):
    """Build an Optuna objective for GANDALF, averaging f1_macro across
    N_INNER_FOLDS inner validation folds via eval_ptab_cv_folds.

    Searches gflu_stages (5-25, capacity) and gflu_dropout (0.0-0.5,
    regularization); learnable_sparsity=False is fixed.
    """

    def objective(trial):
        gflu_stages = trial.suggest_int("gflu_stages", 5, 25)
        gflu_dropout = trial.suggest_float("gflu_dropout", 0.0, 0.5)

        def model_config_fn(common_params):
            return GANDALFConfig(
                gflu_stages=gflu_stages,
                gflu_dropout=gflu_dropout,
                learnable_sparsity=False,
                **common_params,
            )

        return eval_ptab_cv_folds(
            cv_folds, model_config_fn, n_classes, use_class_weight=use_class_weight
        )

    return objective


def _gandalf_model_config_fn(params):
    """Build a GANDALFConfig-constructing closure from a params dict
    (either an Optuna best_params result or GANDALF_DEFAULT_PARAMS) --
    shared by both final-retrain calls in tune_and_evaluate_gandalf_fold.
    """

    def model_config_fn(common_params):
        return GANDALFConfig(
            gflu_stages=params["gflu_stages"],
            gflu_dropout=params["gflu_dropout"],
            learnable_sparsity=False,
            **common_params,
        )

    return model_config_fn


def tune_and_evaluate_gandalf_fold(
    cv_folds,
    X_train_val,
    y_train_val,
    X_test,
    y_test,
    class_names,
    burst,
    correction_type,
    test_fold,
    groups_train_val,
    n_trials=N_TRIALS,
    use_class_weight=True,
):
    """Run one outer test fold for GANDALF: Optuna search scored by the
    mean f1_macro across N_INNER_FOLDS inner validation folds (with
    GANDALF_DEFAULT_PARAMS enqueued as one trial), then retrain the winner
    on the non-test folds (minus the early-stopping subset) and evaluate
    once on the test fold. GANDALF_DEFAULT_PARAMS is retrained and
    evaluated the same way as the test-side default control.

    Returns:
        tuple: (metrics_df, confusion_df, default_metrics_df,
            default_confusion_df, study), with model="GANDALF" /
            "GANDALF_default".
    """
    study_name = f"{burst.lower()}_{correction_type}_testfold{test_fold}"
    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE, multivariate=True)
    study = optuna.create_study(
        direction="maximize", sampler=sampler, study_name=study_name
    )
    study.enqueue_trial(GANDALF_DEFAULT_PARAMS)

    search_start = time.time()
    study.optimize(
        make_gandalf_objective(
            cv_folds, len(class_names), use_class_weight=use_class_weight
        ),
        n_trials=n_trials,
    )
    search_time = time.time() - search_start
    best_params = study.best_params

    default_trial = next(
        (t for t in study.trials if t.params == GANDALF_DEFAULT_PARAMS), None
    )
    default_val_f1_macro = default_trial.value if default_trial is not None else np.nan

    tuned_row, tuned_conf_rows = retrain_and_evaluate_ptab(
        X_train_val,
        y_train_val,
        X_test,
        y_test,
        _gandalf_model_config_fn(best_params),
        best_params,
        class_names,
        groups_train_val,
        use_class_weight=use_class_weight,
    )
    tuned_row.update(
        {
            "model": "GANDALF",
            "val_f1_macro": study.best_value,
            "default_val_f1_macro": default_val_f1_macro,
            "search_time": search_time,
        }
    )
    confusion_data = [{"model": "GANDALF", **row} for row in tuned_conf_rows]

    default_row, default_conf_rows = retrain_and_evaluate_ptab(
        X_train_val,
        y_train_val,
        X_test,
        y_test,
        _gandalf_model_config_fn(GANDALF_DEFAULT_PARAMS),
        GANDALF_DEFAULT_PARAMS,
        class_names,
        groups_train_val,
        use_class_weight=use_class_weight,
    )
    default_row["model"] = "GANDALF_default"
    default_confusion_data = [
        {"model": "GANDALF_default", **row} for row in default_conf_rows
    ]

    results_df = pd.DataFrame([tuned_row])
    confusion_df = pd.DataFrame(confusion_data)
    default_results_df = pd.DataFrame([default_row])
    default_confusion_df = pd.DataFrame(default_confusion_data)

    return results_df, confusion_df, default_results_df, default_confusion_df, study


## GANDALF tuning orchestration, across the 5 outer folds

In [ ]:
def process_parquet_file_gandalf(
    file_path, fold_lookup_all, n_trials=N_TRIALS, use_class_weight=True
):
    """Run GANDALF tuning + final evaluation over the 5 outer folds (see
    notebook intro) for the behaviour target on one file. For each outer
    test fold, cv_folds holds the N_INNER_FOLDS inner (train, val) splits
    of the non-test folds.
    """
    burst = os.path.basename(os.path.dirname(file_path))
    correction_type = os.path.basename(file_path).split("_")[-1].split(".")[0]
    print(f"Processing {correction_type} for {burst}")

    features = pd.read_parquet(file_path)
    id_cols = ["Ind_ID", "new_burst", "burst_id", "behaviour", "fold"]
    m = fold_lookup_all.loc[fold_lookup_all["burst"] == burst.lower(), id_cols]
    df = features.merge(m, on=["Ind_ID", "new_burst"], how="inner").reset_index(
        drop=True
    )

    X_full = df.drop(columns=id_cols)
    # Parent burst per window, for the grouped early-stopping split in the retrains
    groups = (df["Ind_ID"].astype(str) + "_" + df["burst_id"].astype(str)).to_numpy()
    folds = sorted(df["fold"].unique())
    n_folds = len(folds)

    out = {
        "multiclass_metrics": [],
        "multiclass_conf": [],
        "default_metrics": [],
        "default_conf": [],
        "studies": {},
    }
    enc = LabelEncoder()
    y_enc = enc.fit_transform(df["behaviour"])
    class_names = list(enc.classes_)

    for i, test_fold in enumerate(folds):
        val_folds_list = [folds[(i + k) % n_folds] for k in range(1, N_INNER_FOLDS + 1)]
        te = (df["fold"] == test_fold).to_numpy()
        va_masks = [(df["fold"] == vf).to_numpy() for vf in val_folds_list]
        non_test = ~te

        cv_folds = [
            (X_full[~te & ~va], y_enc[~te & ~va], X_full[va], y_enc[va])
            for va in va_masks
        ]

        print(
            f"  Tuning Behaviour classification, test_fold={test_fold}, "
            f"inner val folds={val_folds_list} "
            f"(test={te.sum()} rows, "
            f"inner train/val sizes={[(int((~te & ~va).sum()), int(va.sum())) for va in va_masks]})"
        )
        met, conf, default_met, default_conf, study = tune_and_evaluate_gandalf_fold(
            cv_folds,
            X_full[non_test],
            y_enc[non_test],
            X_full[te],
            y_enc[te],
            class_names,
            burst,
            correction_type,
            test_fold,
            groups_train_val=groups[non_test],
            n_trials=n_trials,
            use_class_weight=use_class_weight,
        )
        for d in (met, conf, default_met, default_conf):
            d["correction_type"] = correction_type
            d["burst"] = burst
            d["target"] = "Behaviour"
            d["test_fold"] = test_fold
            d["val_folds"] = ",".join(str(vf) for vf in val_folds_list)
            d["n_test_samples"] = int(te.sum())
        out["multiclass_metrics"].append(met)
        out["multiclass_conf"].append(conf)
        out["default_metrics"].append(default_met)
        out["default_conf"].append(default_conf)
        out["studies"][test_fold] = study

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    metric_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "gflu_stages",
        "gflu_dropout",
        "val_f1_macro",
        "default_val_f1_macro",
        "roc_auc",
        "accuracy",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "search_time",
        "time",
    ]
    default_metric_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "gflu_stages",
        "gflu_dropout",
        "roc_auc",
        "accuracy",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "time",
    ]
    conf_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "actual_label",
        "predicted_label",
        "count",
    ]
    fin = lambda lst, cols: (
        pd.concat([d for d in lst if not d.empty])[cols]
        if any(not d.empty for d in lst)
        else pd.DataFrame(columns=cols)
    )
    return {
        "multiclass_metrics": fin(out["multiclass_metrics"], metric_columns),
        "multiclass_conf": fin(out["multiclass_conf"], conf_columns),
        "default_metrics": fin(out["default_metrics"], default_metric_columns),
        "default_conf": fin(out["default_conf"], conf_columns),
        "studies": out["studies"],
    }


## Run GANDALF tuning across datasets

In [ ]:
all_results_gandalf = []
for burst, correction_type in DATASETS:
    pq_file_path = (
        f"../data/raw/features/{burst.capitalize()}/"
        f"annotated_features_{burst}_{correction_type}.parquet"
    )
    result = process_parquet_file_gandalf(
        pq_file_path, fold_lookup_all, n_trials=N_TRIALS
    )
    all_results_gandalf.append(result)

multiclass_metrics_gandalf = pd.concat(
    [r["multiclass_metrics"] for r in all_results_gandalf], ignore_index=True
)
multiclass_conf_gandalf = pd.concat(
    [r["multiclass_conf"] for r in all_results_gandalf], ignore_index=True
)
default_metrics_gandalf = pd.concat(
    [r["default_metrics"] for r in all_results_gandalf], ignore_index=True
)
default_conf_gandalf = pd.concat(
    [r["default_conf"] for r in all_results_gandalf], ignore_index=True
)

# Kept in memory (not saved to disk) for the "Inspect best study per
# dataset (GANDALF)" cell near the end of the notebook -- one Optuna
# study per outer fold.
all_studies_gandalf = {
    dataset_key: result["studies"]
    for dataset_key, result in zip(DATASETS, all_results_gandalf)
}


## Category Embedding Optuna search + final evaluation, for one outer test fold

In [ ]:
def make_category_embedding_objective(cv_folds, n_classes, use_class_weight=True):
    """Build an Optuna objective for Category Embedding, averaging
    f1_macro across N_INNER_FOLDS inner validation folds via
    eval_ptab_cv_folds.

    Searches layers (categorical over CATEGORY_EMBEDDING_LAYER_PRESETS,
    capacity) and dropout (0.0-0.5, regularization -- the backbone
    dropout, separate from the head's fixed dropout=0.1).
    """

    def objective(trial):
        layers = trial.suggest_categorical("layers", CATEGORY_EMBEDDING_LAYER_PRESETS)
        dropout = trial.suggest_float("dropout", 0.0, 0.5)

        def model_config_fn(common_params):
            return CategoryEmbeddingModelConfig(
                layers=layers,
                dropout=dropout,
                **common_params,
            )

        return eval_ptab_cv_folds(
            cv_folds, model_config_fn, n_classes, use_class_weight=use_class_weight
        )

    return objective


def _category_embedding_model_config_fn(params):
    """Build a CategoryEmbeddingModelConfig-constructing closure from a
    params dict (either an Optuna best_params result or
    CATEGORY_EMBEDDING_DEFAULT_PARAMS) -- shared by both final-retrain
    calls in tune_and_evaluate_category_embedding_fold.
    """

    def model_config_fn(common_params):
        return CategoryEmbeddingModelConfig(
            layers=params["layers"],
            dropout=params["dropout"],
            **common_params,
        )

    return model_config_fn


def tune_and_evaluate_category_embedding_fold(
    cv_folds,
    X_train_val,
    y_train_val,
    X_test,
    y_test,
    class_names,
    burst,
    correction_type,
    test_fold,
    groups_train_val,
    n_trials=N_TRIALS,
    use_class_weight=True,
):
    """Run one outer test fold for Category Embedding. Same procedure as
    tune_and_evaluate_gandalf_fold, with CATEGORY_EMBEDDING_DEFAULT_PARAMS
    as the enqueued trial and test-side default control.

    Returns:
        tuple: (metrics_df, confusion_df, default_metrics_df,
            default_confusion_df, study), with model="CategoryEmbedding" /
            "CategoryEmbedding_default".
    """
    study_name = f"{burst.lower()}_{correction_type}_testfold{test_fold}"
    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE, multivariate=True)
    study = optuna.create_study(
        direction="maximize", sampler=sampler, study_name=study_name
    )
    study.enqueue_trial(CATEGORY_EMBEDDING_DEFAULT_PARAMS)

    search_start = time.time()
    study.optimize(
        make_category_embedding_objective(
            cv_folds, len(class_names), use_class_weight=use_class_weight
        ),
        n_trials=n_trials,
    )
    search_time = time.time() - search_start
    best_params = study.best_params

    default_trial = next(
        (t for t in study.trials if t.params == CATEGORY_EMBEDDING_DEFAULT_PARAMS),
        None,
    )
    default_val_f1_macro = default_trial.value if default_trial is not None else np.nan

    tuned_row, tuned_conf_rows = retrain_and_evaluate_ptab(
        X_train_val,
        y_train_val,
        X_test,
        y_test,
        _category_embedding_model_config_fn(best_params),
        best_params,
        class_names,
        groups_train_val,
        use_class_weight=use_class_weight,
    )
    tuned_row.update(
        {
            "model": "CategoryEmbedding",
            "val_f1_macro": study.best_value,
            "default_val_f1_macro": default_val_f1_macro,
            "search_time": search_time,
        }
    )
    confusion_data = [{"model": "CategoryEmbedding", **row} for row in tuned_conf_rows]

    default_row, default_conf_rows = retrain_and_evaluate_ptab(
        X_train_val,
        y_train_val,
        X_test,
        y_test,
        _category_embedding_model_config_fn(CATEGORY_EMBEDDING_DEFAULT_PARAMS),
        CATEGORY_EMBEDDING_DEFAULT_PARAMS,
        class_names,
        groups_train_val,
        use_class_weight=use_class_weight,
    )
    default_row["model"] = "CategoryEmbedding_default"
    default_confusion_data = [
        {"model": "CategoryEmbedding_default", **row} for row in default_conf_rows
    ]

    results_df = pd.DataFrame([tuned_row])
    confusion_df = pd.DataFrame(confusion_data)
    default_results_df = pd.DataFrame([default_row])
    default_confusion_df = pd.DataFrame(default_confusion_data)

    return results_df, confusion_df, default_results_df, default_confusion_df, study


## Category Embedding tuning orchestration, across the 5 outer folds

In [ ]:
def process_parquet_file_category_embedding(
    file_path, fold_lookup_all, n_trials=N_TRIALS, use_class_weight=True
):
    """Run Category Embedding tuning + final evaluation over the 5 outer
    folds for the behaviour target on one file. Same structure as
    process_parquet_file_gandalf; only the per-fold trainer and the
    hyperparameter columns (layers, dropout) differ.
    """
    burst = os.path.basename(os.path.dirname(file_path))
    correction_type = os.path.basename(file_path).split("_")[-1].split(".")[0]
    print(f"Processing {correction_type} for {burst}")

    features = pd.read_parquet(file_path)
    id_cols = ["Ind_ID", "new_burst", "burst_id", "behaviour", "fold"]
    m = fold_lookup_all.loc[fold_lookup_all["burst"] == burst.lower(), id_cols]
    df = features.merge(m, on=["Ind_ID", "new_burst"], how="inner").reset_index(
        drop=True
    )

    X_full = df.drop(columns=id_cols)
    # Parent burst per window, for the grouped early-stopping split in the retrains
    groups = (df["Ind_ID"].astype(str) + "_" + df["burst_id"].astype(str)).to_numpy()
    folds = sorted(df["fold"].unique())
    n_folds = len(folds)

    out = {
        "multiclass_metrics": [],
        "multiclass_conf": [],
        "default_metrics": [],
        "default_conf": [],
        "studies": {},
    }
    enc = LabelEncoder()
    y_enc = enc.fit_transform(df["behaviour"])
    class_names = list(enc.classes_)

    for i, test_fold in enumerate(folds):
        val_folds_list = [folds[(i + k) % n_folds] for k in range(1, N_INNER_FOLDS + 1)]
        te = (df["fold"] == test_fold).to_numpy()
        va_masks = [(df["fold"] == vf).to_numpy() for vf in val_folds_list]
        non_test = ~te

        cv_folds = [
            (X_full[~te & ~va], y_enc[~te & ~va], X_full[va], y_enc[va])
            for va in va_masks
        ]

        print(
            f"  Tuning Behaviour classification, test_fold={test_fold}, "
            f"inner val folds={val_folds_list} "
            f"(test={te.sum()} rows, "
            f"inner train/val sizes={[(int((~te & ~va).sum()), int(va.sum())) for va in va_masks]})"
        )
        met, conf, default_met, default_conf, study = (
            tune_and_evaluate_category_embedding_fold(
                cv_folds,
                X_full[non_test],
                y_enc[non_test],
                X_full[te],
                y_enc[te],
                class_names,
                burst,
                correction_type,
                test_fold,
                groups_train_val=groups[non_test],
                n_trials=n_trials,
                use_class_weight=use_class_weight,
            )
        )
        for d in (met, conf, default_met, default_conf):
            d["correction_type"] = correction_type
            d["burst"] = burst
            d["target"] = "Behaviour"
            d["test_fold"] = test_fold
            d["val_folds"] = ",".join(str(vf) for vf in val_folds_list)
            d["n_test_samples"] = int(te.sum())
        out["multiclass_metrics"].append(met)
        out["multiclass_conf"].append(conf)
        out["default_metrics"].append(default_met)
        out["default_conf"].append(default_conf)
        out["studies"][test_fold] = study

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    metric_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "layers",
        "dropout",
        "val_f1_macro",
        "default_val_f1_macro",
        "roc_auc",
        "accuracy",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "search_time",
        "time",
    ]
    default_metric_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "layers",
        "dropout",
        "roc_auc",
        "accuracy",
        "balanced_acc",
        "f1_score",
        "precision",
        "recall",
        "f1_macro",
        "precision_macro",
        "recall_macro",
        "time",
    ]
    conf_columns = [
        "burst",
        "correction_type",
        "target",
        "test_fold",
        "val_folds",
        "n_test_samples",
        "model",
        "actual_label",
        "predicted_label",
        "count",
    ]
    fin = lambda lst, cols: (
        pd.concat([d for d in lst if not d.empty])[cols]
        if any(not d.empty for d in lst)
        else pd.DataFrame(columns=cols)
    )
    return {
        "multiclass_metrics": fin(out["multiclass_metrics"], metric_columns),
        "multiclass_conf": fin(out["multiclass_conf"], conf_columns),
        "default_metrics": fin(out["default_metrics"], default_metric_columns),
        "default_conf": fin(out["default_conf"], conf_columns),
        "studies": out["studies"],
    }


## Run Category Embedding tuning across datasets

In [ ]:
all_results_category_embedding = []
for burst, correction_type in DATASETS:
    pq_file_path = (
        f"../data/raw/features/{burst.capitalize()}/"
        f"annotated_features_{burst}_{correction_type}.parquet"
    )
    result = process_parquet_file_category_embedding(
        pq_file_path, fold_lookup_all, n_trials=N_TRIALS
    )
    all_results_category_embedding.append(result)

multiclass_metrics_category_embedding = pd.concat(
    [r["multiclass_metrics"] for r in all_results_category_embedding],
    ignore_index=True,
)
multiclass_conf_category_embedding = pd.concat(
    [r["multiclass_conf"] for r in all_results_category_embedding], ignore_index=True
)
default_metrics_category_embedding = pd.concat(
    [r["default_metrics"] for r in all_results_category_embedding], ignore_index=True
)
default_conf_category_embedding = pd.concat(
    [r["default_conf"] for r in all_results_category_embedding], ignore_index=True
)

# Kept in memory (not saved to disk) for the "Inspect best study per
# dataset (Category Embedding)" cell near the end of the notebook -- one
# Optuna study per outer fold.
all_studies_category_embedding = {
    dataset_key: result["studies"]
    for dataset_key, result in zip(DATASETS, all_results_category_embedding)
}


## Pool results and inspect the winning hyperparameters per fold

In [ ]:
def pooled_metrics_from_confusion(conf_df, model_name):
    """Reconstruct pooled y_true/y_pred from a summed confusion matrix and
    compute overall metrics on it -- more meaningful than averaging
    per-fold metrics when classes are unevenly represented across folds.
    """
    sub = conf_df[conf_df["model"] == model_name]
    if sub.empty or sub["count"].sum() == 0:
        return {"model": model_name, "n_pooled_test_samples": 0}
    y_true = np.repeat(sub["actual_label"].to_numpy(), sub["count"].to_numpy())
    y_pred = np.repeat(sub["predicted_label"].to_numpy(), sub["count"].to_numpy())


    return {
        "model": model_name,
        "n_pooled_test_samples": len(y_true),
        "accuracy": float((y_true == y_pred).mean()),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "f1_score": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "precision": precision_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_macro": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }


def pooled_summary_per_dataset(conf_df, model_name):
    """Pooled metrics computed separately per (burst, correction_type), so
    datasets are never mixed into a single pool."""
    rows = []
    for (burst, correction_type), grp in conf_df.groupby(["burst", "correction_type"]):
        m = pooled_metrics_from_confusion(grp, model_name)
        m["burst"] = burst
        m["correction_type"] = correction_type
        rows.append(m)
    cols = ["burst", "correction_type"] + [
        c for c in rows[0] if c not in ("burst", "correction_type")
    ]
    return pd.DataFrame(rows)[cols]


def _run_df(name):
    """Return the in-memory DataFrame `name` if that model's cells were
    (re-)run this session, else an empty DataFrame.

    Either model's cells can be run on their own, so not every per-model
    frame necessarily exists in this session. Models not run are skipped,
    and their rows already on disk are left untouched by upsert_csv.
    """
    return globals().get(name, pd.DataFrame())


# Combine this session's GANDALF and/or Category Embedding results into
# single frames; they are merged with the rows already on disk in the save
# cell below.
multiclass_metrics_run = pd.concat(
    [
        df
        for df in (
            _run_df("multiclass_metrics_gandalf"),
            _run_df("multiclass_metrics_category_embedding"),
        )
        if not df.empty
    ],
    ignore_index=True,
)
multiclass_conf_run = pd.concat(
    [
        df
        for df in (
            _run_df("multiclass_conf_gandalf"),
            _run_df("multiclass_conf_category_embedding"),
        )
        if not df.empty
    ],
    ignore_index=True,
)
default_metrics_run = pd.concat(
    [
        df
        for df in (
            _run_df("default_metrics_gandalf"),
            _run_df("default_metrics_category_embedding"),
        )
        if not df.empty
    ],
    ignore_index=True,
)
default_conf_run = pd.concat(
    [
        df
        for df in (
            _run_df("default_conf_gandalf"),
            _run_df("default_conf_category_embedding"),
        )
        if not df.empty
    ],
    ignore_index=True,
)

# Derived from the models present in this run (see _run_df's docstring).
MODEL_NAMES = sorted(multiclass_conf_run["model"].unique())
print(f"Models pooled this run: {MODEL_NAMES}")

pooled_multiclass_summary = pd.concat(
    [pooled_summary_per_dataset(multiclass_conf_run, m) for m in MODEL_NAMES],
    ignore_index=True,
)
print("\nPooled tuned multiclass metrics per dataset:")
print(pooled_multiclass_summary)

# Per-fold test performance. val_f1_macro vs default_val_f1_macro is the
# validation-side default control. Model-specific hyperparameter columns
# are omitted from this print but kept in multiclass_metrics_run and the
# saved CSV.
per_fold_cols = [
    "burst",
    "correction_type",
    "test_fold",
    "val_folds",
    "n_test_samples",
    "model",
    "val_f1_macro",
    "default_val_f1_macro",
    "accuracy",
    "balanced_acc",
    "f1_macro",
    "search_time",
    "time",
]
per_fold_multiclass = multiclass_metrics_run[per_fold_cols].sort_values(
    ["burst", "correction_type", "model", "test_fold"]
)
print(
    "\nPer-fold tuned multiclass results (final test metrics; see "
    "multiclass_metrics_run for the full per-model hyperparameter columns):"
)
print(per_fold_multiclass)

# In how many outer folds did the search beat the default on the inner
# validation folds, and by how much on average (validation side only).
beats_default = (
    per_fold_multiclass["val_f1_macro"] > per_fold_multiclass["default_val_f1_macro"]
)
print(
    f"\nSearch beat the default on validation in {beats_default.sum()}/{len(beats_default)} "
    f"outer folds across {MODEL_NAMES} (mean val_f1_macro - default_val_f1_macro = "
    f"{(per_fold_multiclass['val_f1_macro'] - per_fold_multiclass['default_val_f1_macro']).mean():.4f})"
)

# Default control on the test fold: pooled tuned vs pooled default
# performance, both retrained and evaluated on the same outer folds.
DEFAULT_MODEL_NAMES = sorted(default_conf_run["model"].unique())
pooled_default_summary = pd.concat(
    [pooled_summary_per_dataset(default_conf_run, m) for m in DEFAULT_MODEL_NAMES],
    ignore_index=True,
)
print("\nPooled DEFAULT (test-fold) metrics per dataset:")
print(pooled_default_summary)

# Pair each tuned model with its own "<model>_default" row (merging on
# [burst, correction_type] alone would give a cross product).
test_gain = pooled_multiclass_summary.assign(
    model_base=pooled_multiclass_summary["model"]
).merge(
    pooled_default_summary.assign(
        model_base=pooled_default_summary["model"].str.replace(
            "_default", "", regex=False
        )
    ),
    on=["burst", "correction_type", "model_base"],
    suffixes=("_tuned", "_default"),
)[["burst", "correction_type", "model_base", "f1_macro_tuned", "f1_macro_default"]]
test_gain["f1_macro_gain"] = test_gain["f1_macro_tuned"] - test_gain["f1_macro_default"]
print("\nTest-set gain (tuned - default, pooled f1_macro), per model:")
print(test_gain)

## Inspect best study per dataset (GANDALF)

Run after all outer folds are complete. For the chosen dataset, finds the
outer fold with the best validation `f1_macro` and shows its Optuna study
summary and plots.

Change `INSPECT_DATASET_GANDALF` to any tuple from `DATASETS` to look at a
different dataset.

In [ ]:
# Change this to inspect a different dataset -- must be one of the tuples
# in DATASETS.
INSPECT_DATASET_GANDALF = DATASETS[0]

dataset_rows_gandalf = multiclass_metrics_gandalf[
    (multiclass_metrics_gandalf["burst"].str.lower() == INSPECT_DATASET_GANDALF[0])
    & (multiclass_metrics_gandalf["correction_type"] == INSPECT_DATASET_GANDALF[1])
]
best_row_gandalf = dataset_rows_gandalf.sort_values(
    "val_f1_macro", ascending=False
).iloc[0]
best_test_fold_gandalf = best_row_gandalf["test_fold"]
study_gandalf = all_studies_gandalf[INSPECT_DATASET_GANDALF][best_test_fold_gandalf]

complete_trials_gandalf = [
    t for t in study_gandalf.trials if t.state == optuna.trial.TrialState.COMPLETE
]
print(
    f"Best-performing outer fold for {INSPECT_DATASET_GANDALF} "
    f"(by validation f1_macro across its {len(dataset_rows_gandalf)} outer folds): "
    f"test_fold={best_test_fold_gandalf}"
)
print(
    f"  # trials (completed) : {len(complete_trials_gandalf)}/{len(study_gandalf.trials)}"
)
print(f"  Best value            : {study_gandalf.best_value:.4f}")
print(f"  Best params           : {study_gandalf.best_params}")
print(f"  Default params        : {GANDALF_DEFAULT_PARAMS}")
print(f"  Default value         : {best_row_gandalf['default_val_f1_macro']:.4f}")
print(
    f"  Gain vs default       : {study_gandalf.best_value - best_row_gandalf['default_val_f1_macro']:+.4f}"
)

for plot_fn in [
    optuna.visualization.plot_optimization_history,
    optuna.visualization.plot_param_importances,
    optuna.visualization.plot_slice,
    optuna.visualization.plot_parallel_coordinate,
]:
    try:
        plot_fn(study_gandalf).show()
    except Exception as e:
        print(f"  Could not generate {plot_fn.__name__}: {e!s}")

## Inspect best study per dataset (Category Embedding)

Same as the GANDALF cell above.

In [ ]:
# Change this to inspect a different dataset -- must be one of the tuples
# in DATASETS.
INSPECT_DATASET_CE = DATASETS[0]

dataset_rows_ce = multiclass_metrics_category_embedding[
    (
        multiclass_metrics_category_embedding["burst"].str.lower()
        == INSPECT_DATASET_CE[0]
    )
    & (
        multiclass_metrics_category_embedding["correction_type"]
        == INSPECT_DATASET_CE[1]
    )
]
best_row_ce = dataset_rows_ce.sort_values("val_f1_macro", ascending=False).iloc[0]
best_test_fold_ce = best_row_ce["test_fold"]
study_ce = all_studies_category_embedding[INSPECT_DATASET_CE][best_test_fold_ce]

complete_trials_ce = [
    t for t in study_ce.trials if t.state == optuna.trial.TrialState.COMPLETE
]
print(
    f"Best-performing outer fold for {INSPECT_DATASET_CE} "
    f"(by validation f1_macro across its {len(dataset_rows_ce)} outer folds): "
    f"test_fold={best_test_fold_ce}"
)
print(f"  # trials (completed) : {len(complete_trials_ce)}/{len(study_ce.trials)}")
print(f"  Best value            : {study_ce.best_value:.4f}")
print(f"  Best params           : {study_ce.best_params}")
print(f"  Default params        : {CATEGORY_EMBEDDING_DEFAULT_PARAMS}")
print(f"  Default value         : {best_row_ce['default_val_f1_macro']:.4f}")
print(
    f"  Gain vs default       : {study_ce.best_value - best_row_ce['default_val_f1_macro']:+.4f}"
)

for plot_fn in [
    optuna.visualization.plot_optimization_history,
    optuna.visualization.plot_param_importances,
    optuna.visualization.plot_slice,
    optuna.visualization.plot_parallel_coordinate,
]:
    try:
        plot_fn(study_ce).show()
    except Exception as e:
        print(f"  Could not generate {plot_fn.__name__}: {e!s}")


## Helper: merge new results into existing output CSVs

In [ ]:
def upsert_csv(new_df, path, key_cols):
    """Merge new_df's rows into the CSV at path (creating it if it
    doesn't exist yet), keeping new_df's rows wherever key_cols collide
    with what's already on disk.

    This lets GANDALF and Category Embedding be run independently without
    dropping the other model's rows from the shared output files. Called
    from the save cell, once per output file.
    """
    if os.path.exists(path):
        existing = pd.read_csv(path)
        combined = pd.concat([existing, new_df], ignore_index=True)
        combined = combined.drop_duplicates(subset=key_cols, keep="last")
    else:
        combined = new_df
    combined.to_csv(path, index=False)
    return combined


## Save outputs

In [ ]:
# Merges this session's results (built in the pooling cell) into the
# output CSVs via upsert_csv, so rows for models not run this session are
# kept. Only the behaviour target is saved.
METRIC_KEY = ["burst", "correction_type", "target", "test_fold", "model"]
CONF_KEY = [
    "burst",
    "correction_type",
    "target",
    "test_fold",
    "model",
    "actual_label",
    "predicted_label",
]

upsert_csv(
    multiclass_metrics_run,
    "../data/output/comparison_behaviour/behaviour_features_dl_tuned_metrics.csv",
    METRIC_KEY,
)
upsert_csv(
    multiclass_conf_run,
    "../data/output/comparison_behaviour/behaviour_features_dl_tuned_confusion.csv",
    CONF_KEY,
)

# Default-hyperparameter control (test-fold retrain), saved to separate
# files so the tuned metrics/confusion files read by
# 12b_publication_plots_fig3.R contain only tuned rows.
upsert_csv(
    default_metrics_run,
    "../data/output/comparison_behaviour/behaviour_features_dl_tuned_default_control_metrics.csv",
    METRIC_KEY,
)
upsert_csv(
    default_conf_run,
    "../data/output/comparison_behaviour/behaviour_features_dl_tuned_default_control_confusion.csv",
    CONF_KEY,
)